# Setup

In [ ]:
import time
from collections.abc import Callable
from functools import partial
from pathlib import Path
from typing import NamedTuple

import jax.numpy as jnp
import jaxopt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from jax import block_until_ready, jit, lax, random, vmap

In [ ]:
config = {
    "output_path": Path("./results/"),
}

# Algorithms

In [ ]:
class SwarmState(NamedTuple):
    positions: np.ndarray
    velocities: np.ndarray
    p_best_pos: np.ndarray
    p_best_fit: np.ndarray
    g_best_pos: np.ndarray
    g_best_fit: np.ndarray
    rng: np.random.Generator
    history: np.ndarray

In [ ]:
class JaxSwarmState(NamedTuple):
    positions: jnp.ndarray
    velocities: jnp.ndarray
    p_best_pos: jnp.ndarray
    p_best_fit: jnp.ndarray
    g_best_pos: jnp.ndarray
    g_best_fit: jnp.ndarray
    rng: random.PRNGKey

## PSO (NumPy)

In [ ]:
def pso(
    objective_fn: callable,
    bounds: tuple,
    num_dims: int,
    num_particles: int,
    max_iters: int,
    c1: float,
    c2: float,
    w: float,
    seed: int,
    **_: any,
) -> tuple:
    lower, upper = bounds
    rng = np.random.default_rng(seed)

    init_positions = rng.uniform(lower, upper, (num_particles, num_dims))
    init_velocities = np.zeros((num_particles, num_dims))
    init_fitness = np.array([objective_fn(position) for position in init_positions])

    best_idx = np.argmin(init_fitness)
    g_best_pos = init_positions[best_idx]
    g_best_fit = init_fitness[best_idx]

    history = np.zeros(max_iters)
    history[0] = g_best_fit

    swarm_state = SwarmState(
        positions=init_positions,
        velocities=init_velocities,
        p_best_pos=init_positions,
        p_best_fit=init_fitness,
        g_best_pos=g_best_pos,
        g_best_fit=g_best_fit,
        rng=rng,
        history=history,
    )

    for i in range(max_iters):
        r1 = swarm_state.rng.random((num_particles, num_dims))
        r2 = swarm_state.rng.random((num_particles, num_dims))

        inertia = w * swarm_state.velocities
        cognitive = c1 * r1 * (swarm_state.p_best_pos - swarm_state.positions)
        social = c2 * r2 * (swarm_state.g_best_pos - swarm_state.positions)

        new_velocities = inertia + cognitive + social
        new_positions = swarm_state.positions + new_velocities
        new_positions = np.clip(new_positions, lower, upper)

        new_fitness = np.array([objective_fn(pos) for pos in new_positions])

        improved = new_fitness < swarm_state.p_best_fit
        mask = improved[:, None]
        new_p_best_pos = np.where(mask, new_positions, swarm_state.p_best_pos)
        new_p_best_fit = np.where(improved, new_fitness, swarm_state.p_best_fit)

        current_g_best_idx = np.argmin(new_p_best_fit)
        current_g_best_fit = new_p_best_fit[current_g_best_idx]
        global_improved = current_g_best_fit < swarm_state.g_best_fit
        new_g_best_pos = np.where(
            global_improved,
            new_p_best_pos[current_g_best_idx],
            swarm_state.g_best_pos,
        )
        new_g_best_fit = np.where(
            global_improved,
            current_g_best_fit,
            swarm_state.g_best_fit,
        )

        new_history = swarm_state.history
        new_history[i] = new_g_best_fit

        swarm_state = SwarmState(
            positions=new_positions,
            velocities=new_velocities,
            p_best_pos=new_p_best_pos,
            p_best_fit=new_p_best_fit,
            g_best_pos=new_g_best_pos,
            g_best_fit=new_g_best_fit,
            rng=swarm_state.rng,
            history=new_history,
        )

    return swarm_state.g_best_pos, swarm_state.g_best_fit, swarm_state.history

## PSO (JAX)

In [ ]:
@partial(
    jit,
    static_argnames=(
        "objective_fn",
        "num_dims",
        "num_particles",
        "max_iters",
    ),
)
def pso_jax(
    objective_fn: Callable,
    bounds: tuple,
    num_dims: int,
    num_particles: int,
    max_iters: int,
    c1: float,
    c2: float,
    w: float,
    seed: random.PRNGKey,
    **_: any,
) -> tuple:
    key = seed
    lower, upper = jnp.array(bounds[0]), jnp.array(bounds[1])
    k_pos, k_vel, k_state = random.split(key, 3)

    search_range = upper - lower
    velocity_scale = 0.1
    limit = search_range * velocity_scale

    init_positions = random.uniform(k_pos, (num_particles, num_dims), minval=lower, maxval=upper)
    init_velocities = random.uniform(k_vel, (num_particles, num_dims), minval=-limit, maxval=limit)
    init_fitness = vmap(objective_fn)(init_positions)

    best_idx = jnp.argmin(init_fitness)
    g_best_pos = init_positions[best_idx]
    g_best_fit = init_fitness[best_idx]

    initial_state = JaxSwarmState(
        positions=init_positions,
        velocities=init_velocities,
        p_best_pos=init_positions,
        p_best_fit=init_fitness,
        g_best_pos=g_best_pos,
        g_best_fit=g_best_fit,
        rng=k_state,
    )

    def update_step(swarm_state: JaxSwarmState, _: any) -> tuple:
        k1, k2, k_next = random.split(swarm_state.rng, 3)
        r1 = random.uniform(k1, (num_particles, num_dims))
        r2 = random.uniform(k2, (num_particles, num_dims))

        inertia = w * swarm_state.velocities
        cognitive = c1 * r1 * (swarm_state.p_best_pos - swarm_state.positions)
        social = c2 * r2 * (swarm_state.g_best_pos - swarm_state.positions)

        new_velocities = inertia + cognitive + social
        new_positions = swarm_state.positions + new_velocities
        new_positions = jnp.clip(new_positions, lower, upper)

        new_fitness = vmap(objective_fn)(new_positions)

        improved = new_fitness < swarm_state.p_best_fit

        new_p_best_pos = jnp.where(improved[:, None], new_positions, swarm_state.p_best_pos)
        new_p_best_fit = jnp.where(improved, new_fitness, swarm_state.p_best_fit)

        current_g_best_idx = jnp.argmin(new_p_best_fit)
        new_g_best_pos = new_p_best_pos[current_g_best_idx]
        new_g_best_fit = new_p_best_fit[current_g_best_idx]

        global_improved = new_g_best_fit < swarm_state.g_best_fit

        final_g_best_pos = jnp.where(global_improved, new_g_best_pos, swarm_state.g_best_pos)
        final_g_best_fit = jnp.where(global_improved, new_g_best_fit, swarm_state.g_best_fit)

        next_state = JaxSwarmState(
            positions=new_positions,
            velocities=new_velocities,
            p_best_pos=new_p_best_pos,
            p_best_fit=new_p_best_fit,
            g_best_pos=final_g_best_pos,
            g_best_fit=final_g_best_fit,
            rng=k_next,
        )

        return next_state, final_g_best_fit

    final_state, history = lax.scan(update_step, initial_state, jnp.arange(max_iters))
    full_history = jnp.concatenate([jnp.array([initial_state.g_best_fit]), history])

    return final_state.g_best_pos, final_state.g_best_fit, full_history

## PSOX (NumPy + SciPy)

In [ ]:
def numerical_gradient(objective_fn: callable, x: np.ndarray, epsilon: float = 1e-8) -> np.ndarray:
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += epsilon
        x_minus = x.copy()
        x_minus[i] -= epsilon
        grad[i] = (objective_fn(x_plus) - objective_fn(x_minus)) / (2 * epsilon)
    return grad


def gradient_descent(
    objective_fn: callable,
    initial_pos: np.ndarray,
    bounds: tuple,
    eta: float,
    steps: int,
) -> tuple:
    lower, upper = bounds
    current_pos = initial_pos.copy()

    for _ in range(steps):
        grad = numerical_gradient(objective_fn, current_pos)
        current_pos = current_pos - eta * grad
        current_pos = np.clip(current_pos, lower, upper)

    final_fit = objective_fn(current_pos)
    return current_pos, final_fit


def gdpso(
    objective_fn: callable,
    bounds: tuple,
    num_dims: int,
    num_particles: int,
    max_iters: int,
    c1: float,
    c2: float,
    w: float,
    seed: int,
    eta: float,
    steps: int,
    gd_interval: int,
    **_: any,
) -> tuple:
    lower, upper = bounds
    rng = np.random.default_rng(seed)

    init_positions = rng.uniform(lower, upper, (num_particles, num_dims))
    init_velocities = np.zeros((num_particles, num_dims))
    init_fitness = np.array([objective_fn(position) for position in init_positions])

    best_idx = np.argmin(init_fitness)
    g_best_pos = init_positions[best_idx]
    g_best_fit = init_fitness[best_idx]

    history = np.zeros(max_iters)
    history[0] = g_best_fit

    swarm_state = SwarmState(
        positions=init_positions,
        velocities=init_velocities,
        p_best_pos=init_positions,
        p_best_fit=init_fitness,
        g_best_pos=g_best_pos,
        g_best_fit=g_best_fit,
        rng=rng,
        history=history,
    )

    for i in range(max_iters):
        r1 = swarm_state.rng.random((num_particles, num_dims))
        r2 = swarm_state.rng.random((num_particles, num_dims))

        inertia = w * swarm_state.velocities
        cognitive = c1 * r1 * (swarm_state.p_best_pos - swarm_state.positions)
        social = c2 * r2 * (swarm_state.g_best_pos - swarm_state.positions)

        new_velocities = inertia + cognitive + social
        new_positions = swarm_state.positions + new_velocities
        new_positions = np.clip(new_positions, lower, upper)

        new_fitness = np.array([objective_fn(pos) for pos in new_positions])

        improved = new_fitness < swarm_state.p_best_fit
        mask = improved[:, None]
        new_p_best_pos = np.where(mask, new_positions, swarm_state.p_best_pos)
        new_p_best_fit = np.where(improved, new_fitness, swarm_state.p_best_fit)

        current_g_best_idx = np.argmin(new_p_best_fit)
        current_g_best_fit = new_p_best_fit[current_g_best_idx]
        global_improved = current_g_best_fit < swarm_state.g_best_fit

        candidate_g_pos = np.where(
            global_improved,
            new_p_best_pos[current_g_best_idx],
            swarm_state.g_best_pos,
        )
        candidate_g_fit = np.where(
            global_improved,
            current_g_best_fit,
            swarm_state.g_best_fit,
        )

        if i % gd_interval == 0:
            gradient_g_pos, gradient_g_fit = gradient_descent(
                objective_fn,
                candidate_g_pos,
                bounds,
                eta,
                steps,
            )

            gd_improved = gradient_g_fit < candidate_g_fit
            final_g_pos = gradient_g_pos if gd_improved else candidate_g_pos
            final_g_fit = gradient_g_fit if gd_improved else candidate_g_fit

            if gd_improved and (final_g_fit < swarm_state.g_best_fit):
                new_p_best_pos[current_g_best_idx] = final_g_pos
                new_p_best_fit[current_g_best_idx] = final_g_fit
        else:
            final_g_pos = candidate_g_pos
            final_g_fit = candidate_g_fit

        new_history = swarm_state.history
        new_history[i] = final_g_fit

        swarm_state = SwarmState(
            positions=new_positions,
            velocities=new_velocities,
            p_best_pos=new_p_best_pos,
            p_best_fit=new_p_best_fit,
            g_best_pos=final_g_pos,
            g_best_fit=final_g_fit,
            rng=swarm_state.rng,
            history=new_history,
        )

    return swarm_state.g_best_pos, swarm_state.g_best_fit, swarm_state.history


## PSOX (JAX + JAXopt)

In [ ]:
class GradientState(NamedTuple):
    current_pos: jnp.ndarray


@partial(
    jit,
    static_argnames=(
        "objective_fn",
        "num_dims",
        "num_particles",
        "max_iters",
        "steps",
        "gd_interval",
        "eta",
    ),
)
def gdpso_jax(
    objective_fn: Callable,
    bounds: tuple,
    num_dims: int,
    num_particles: int,
    max_iters: int,
    c1: float,
    c2: float,
    w: float,
    seed: random.PRNGKey,
    eta: float,
    steps: int,
    gd_interval: int,
    **_: any,
) -> tuple:
    key = seed
    lower, upper = jnp.array(bounds[0]), jnp.array(bounds[1])
    k_pos, k_vel, k_state = random.split(key, 3)

    search_range = upper - lower
    velocity_scale = 0.1
    limit = search_range * velocity_scale

    init_positions = random.uniform(k_pos, (num_particles, num_dims), minval=lower, maxval=upper)
    init_velocities = random.uniform(k_vel, (num_particles, num_dims), minval=-limit, maxval=limit)
    init_fitness = vmap(objective_fn)(init_positions)

    best_idx = jnp.argmin(init_fitness)
    g_best_pos = init_positions[best_idx]
    g_best_fit = init_fitness[best_idx]

    initial_state = JaxSwarmState(
        positions=init_positions,
        velocities=init_velocities,
        p_best_pos=init_positions,
        p_best_fit=init_fitness,
        g_best_pos=g_best_pos,
        g_best_fit=g_best_fit,
        rng=k_state,
    )

    gd_solver = jaxopt.GradientDescent(
        fun=objective_fn,
        stepsize=eta,
        implicit_diff=False,
    )

    def update_step(swarm_state: JaxSwarmState, i: int) -> tuple:
        k1, k2, k_next = random.split(swarm_state.rng, 3)
        r1 = random.uniform(k1, (num_particles, num_dims))
        r2 = random.uniform(k2, (num_particles, num_dims))

        inertia = w * swarm_state.velocities
        cognitive = c1 * r1 * (swarm_state.p_best_pos - swarm_state.positions)
        social = c2 * r2 * (swarm_state.g_best_pos - swarm_state.positions)

        new_velocities = inertia + cognitive + social
        new_positions = swarm_state.positions + new_velocities
        new_positions = jnp.clip(new_positions, lower, upper)

        new_fitness = vmap(objective_fn)(new_positions)

        improved = new_fitness < swarm_state.p_best_fit

        new_p_best_pos = jnp.where(improved[:, None], new_positions, swarm_state.p_best_pos)
        new_p_best_fit = jnp.where(improved, new_fitness, swarm_state.p_best_fit)

        current_g_best_idx = jnp.argmin(new_p_best_fit)
        current_g_best_fit = new_p_best_fit[current_g_best_idx]

        global_improved = current_g_best_fit < swarm_state.g_best_fit

        candidate_g_pos = jnp.where(
            global_improved,
            new_p_best_pos[current_g_best_idx],
            swarm_state.g_best_pos,
        )

        candidate_g_fit = jnp.where(global_improved, current_g_best_fit, swarm_state.g_best_fit)

        def apply_gd(_: None) -> tuple:
            init_gd_state = gd_solver.init_state(candidate_g_pos)

            def gd_step(carry: tuple, _: None) -> tuple:
                pos, state = carry

                next_pos, next_state = gd_solver.update(pos, state)
                next_pos = jnp.clip(next_pos, upper, lower)

                return (next_pos, next_state), None

            (final_pos, _), _ = lax.scan(
                gd_step,
                (candidate_g_pos, init_gd_state),
                None,
                steps,
            )

            final_fit = objective_fn(final_pos)
            return final_pos, final_fit

        def skip_gd(_: None) -> tuple:
            return candidate_g_pos, candidate_g_fit

        gradient_g_pos, gradient_g_fit = lax.cond(
            i % gd_interval == 0,
            apply_gd,
            skip_gd,
            None,
        )

        gd_improved = gradient_g_fit < candidate_g_fit
        final_g_pos = jnp.where(gd_improved, gradient_g_pos, candidate_g_pos)
        final_g_fit = jnp.where(gd_improved, gradient_g_fit, candidate_g_fit)

        any_improvement = final_g_fit < swarm_state.g_best_fit

        target_idx = current_g_best_idx

        mask_winner = (jnp.arange(num_particles) == target_idx)[:, None]
        should_update_mask = (gd_improved & any_improvement) & mask_winner

        final_p_best_pos = jnp.where(
            should_update_mask,
            final_g_pos,
            new_p_best_pos,
        )

        final_p_best_fit = jnp.where(
            (gd_improved & any_improvement) & (jnp.arange(num_particles) == target_idx),
            final_g_fit,
            new_p_best_fit,
        )

        next_state = JaxSwarmState(
            positions=new_positions,
            velocities=new_velocities,
            p_best_pos=final_p_best_pos,
            p_best_fit=final_p_best_fit,
            g_best_pos=final_g_pos,
            g_best_fit=final_g_fit,
            rng=k_next,
        )

        return next_state, final_g_fit

    final_state, history = lax.scan(update_step, initial_state, jnp.arange(max_iters))
    full_history = jnp.concatenate([jnp.array([initial_state.g_best_fit]), history])

    return final_state.g_best_pos, final_state.g_best_fit, full_history

# Visualizations

In [ ]:
def _save_figure(fig: plt.Figure, filename: str, config: dict) -> None:
    save_path = config["output_path"] / filename
    fig.savefig(save_path, bbox_inches="tight", format="pdf", dpi=300)
    plt.close(fig)


def plot_execution_time(df: pd.DataFrame, config: dict) -> None:
    benchmarks = df["Benchmark"].unique()
    algorithms = df["Algorithm"].unique()
    dimensions = df["Dimension"].unique()

    colors = ["#909090", "#0056AD", "#6D87A1", "#0080FF"]

    plt.rcParams.update(
        {
            "font.size": 11,
            "axes.titlesize": 11,
            "axes.labelsize": 11,
            "xtick.labelsize": 11,
            "ytick.labelsize": 11,
            "legend.fontsize": 11,
        },
    )

    fig, axes = plt.subplots(2, 2, figsize=(5, 4), sharex=True, sharey=True)
    axes_flattened = axes.flatten()

    lines = []
    labels = []

    for idx_ax, (ax, benchmark) in enumerate(zip(axes_flattened, benchmarks)):
        df_benchmark = df[df["Benchmark"] == benchmark]

        for idx_algo, algorithm in enumerate(algorithms):
            df_subset = df_benchmark[df_benchmark["Algorithm"] == algorithm]
            mean = df_subset["Mean of Execution Times (s)"]
            std = df_subset["Standard Deviation of Execution Times (s)"]

            (line,) = ax.plot(
                dimensions,
                mean,
                label=algorithm,
                marker="o",
                color=colors[idx_algo],
                markersize=6,
                linewidth=2,
            )
            ax.fill_between(dimensions, mean - std, mean + std, color=colors[idx_algo], alpha=0.2)

            if idx_ax == 0:
                lines.append(line)
                labels.append(algorithm)

        ax.set_yscale("log")
        ax.set_title(f"{benchmark}")
        sns.despine(ax=ax, left=True)

    fig.supxlabel("Dimensão")
    fig.supylabel("Tempo de Execução (s)")

    fig.legend(
        lines,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.05),
        ncol=len(algorithms),
        frameon=False,
    )

    plt.tight_layout()
    _save_figure(fig, "execution_time_plot.pdf", config)


def plot_convergence(df: pd.DataFrame, config: dict) -> None:
    plt.rcParams.update(
        {
            "font.size": 13,
            "axes.titlesize": 13,
            "axes.labelsize": 13,
            "xtick.labelsize": 13,
            "ytick.labelsize": 13,
            "legend.fontsize": 13,
        },
    )

    benchmarks = df["Benchmark"].unique()
    dimensions = df["Dimension"].unique()
    algorithms = df["Algorithm"].unique()

    colors = ["#909090", "#0056AD", "#6D87A1", "#0080FF"]

    fig, axes = plt.subplots(4, 4, figsize=(12, 12), sharex=True)

    lines = []
    labels = []

    for i, benchmark in enumerate(benchmarks):
        for j, dimension in enumerate(dimensions):
            ax = axes[i, j]
            df_subset = df[(df["Dimension"] == dimension) & (df["Benchmark"] == benchmark)]

            for k, (_, row) in enumerate(df_subset.iterrows()):
                mean_history = jnp.array(row["Mean Fitness History"])
                std_history = jnp.array(row["Std Fitness History"])
                iterations = range(len(mean_history))

                (line,) = ax.plot(
                    iterations,
                    mean_history,
                    label=row["Algorithm"],
                    color=colors[k],
                    linewidth=2,
                )
                ax.fill_between(
                    iterations,
                    mean_history - std_history,
                    mean_history + std_history,
                    alpha=0.2,
                    color=colors[k],
                )

                if i == 0 and j == 0:
                    lines.append(line)
                    labels.append(row["Algorithm"])

            sns.despine(ax=ax, left=True)
            ax.set_title(f"{benchmark} (d = {dimension})")

    fig.supxlabel("Iteração")
    fig.supylabel("Fitness")

    fig.legend(
        lines,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.03),
        ncol=len(algorithms),
        frameon=False,
    )

    plt.tight_layout()
    _save_figure(fig, "convergence_plot.pdf", config)


def _generate_comparison_table(
    df: pd.DataFrame,
    config: dict,
    mean_col: str,
    std_col: str,
    output_filename: str,
    caption: str,
    label: str,
) -> None:
    df_proc = df.copy()

    df_proc["formatted"] = (
        df_proc[mean_col].map("{:.2e}".format) + r" $\pm$ " + df_proc[std_col].map("{:.2e}".format)
    )

    df_pivot = df_proc.pivot_table(
        index=["Benchmark", "Dimension"],
        columns="Algorithm",
        values=["formatted", mean_col],
        aggfunc="first",
    )

    means = df_pivot[mean_col].fillna(float("inf"))

    display_df = df_pivot["formatted"].copy()

    for idx in display_df.index:
        row_means = means.loc[idx]
        best_algo = row_means.idxmin()
        if pd.notna(display_df.loc[idx, best_algo]):
            display_df.loc[idx, best_algo] = r"\textbf{" + display_df.loc[idx, best_algo] + "}"

    display_df = display_df.reset_index()

    display_df["Benchmark"] = display_df["Benchmark"].apply(lambda x: f"\\textit{{{x}}}")

    for col in display_df.columns:
        if col not in ["Benchmark", "Dimension"]:
            display_df = display_df.rename(columns={col: f"\\textit{{{col}}}"})

    output_path = config["output_path"] / output_filename

    num_algo_cols = len(display_df.columns) - 2
    column_format = "ll" + "c" * num_algo_cols

    latex_code = display_df.style.hide(axis="index").to_latex(
        column_format=column_format,
        hrules=True,
        caption=caption,
        label=label,
        position="h",
    )

    with output_path.open("w") as f:
        f.write(latex_code)


def create_convergence_table(df: pd.DataFrame, config: dict) -> None:
    _generate_comparison_table(
        df=df,
        config=config,
        mean_col="Mean of Fitness",
        std_col="Standard Deviation of Fitness",
        output_filename="convergence_table.tex",
        caption=r"Convergence comparison (Mean Fitness $\pm$ Std Dev). Best results in bold.",
        label="tab:convergence",
    )


def create_execution_time_table(df: pd.DataFrame, config: dict) -> None:
    _generate_comparison_table(
        df=df,
        config=config,
        mean_col="Mean of Execution Times (s)",
        std_col="Standard Deviation of Execution Times (s)",
        output_filename="execution_time_table.tex",
        caption="Execution time comparison in seconds.",
        label="tab:execution_time",
    )


def generate_visualizations(df: pd.DataFrame, config: dict) -> None:
    print("Plotting convergence...")
    plot_convergence(df, config)

    print("Plotting execution time...")
    plot_execution_time(df, config)

    print("Creating convergence table (LaTeX)...")
    create_convergence_table(df, config)

    print("Creating execution time table (LaTeX)...")
    create_execution_time_table(df, config)

# Benchmark Functions

## NumPy

In [ ]:
def schwefel_np(x: np.ndarray) -> float:
    n = x.shape[0]
    sum_term = np.sum(x * np.sin(np.sqrt(np.abs(x))))
    return 418.9829 * n - sum_term


def rastrigin_np(x: np.ndarray) -> float:
    n = x.shape[0]
    return 10 * n + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))


def sphere_np(x: np.ndarray) -> float:
    return np.sum(x**2)


def elliptic_np(x: np.ndarray) -> float:
    n = x.shape[0]
    i = np.arange(n)
    coeffs = (1e6) ** (i / (n - 1))
    return np.sum(coeffs * x**2)

## JAX

In [ ]:
@jit
def schwefel_jax(x: jnp.ndarray) -> jnp.ndarray:
    n = x.shape[0]
    sum_term = jnp.sum(x * jnp.sin(jnp.sqrt(jnp.abs(x))))
    return 418.9829 * n - sum_term


@jit
def rastrigin_jax(x: jnp.ndarray) -> jnp.ndarray:
    n = x.shape[0]
    return 10 * n + jnp.sum(x**2 - 10 * jnp.cos(2 * jnp.pi * x))


@jit
def sphere_jax(x: jnp.ndarray) -> jnp.ndarray:
    return jnp.sum(x**2)


@jit
def elliptic_jax(x: jnp.ndarray) -> jnp.ndarray:
    n = x.shape[0]
    i = jnp.arange(n)

    coeffs = (1e6) ** (i / (n - 1))
    return jnp.sum(coeffs * x**2)

# Configurataion

In [ ]:
BENCHMARKS = {
    "Schwefel": {
        "bounds": (-500, 500),
        "NumPy": schwefel_np,
        "JAX": schwefel_jax,
    },
    "Rastrigin": {
        "bounds": (-5.12, 5.12),
        "NumPy": rastrigin_np,
        "JAX": rastrigin_jax,
    },
    "Elliptic": {
        "bounds": (-5.0, 10.0),
        "NumPy": elliptic_np,
        "JAX": elliptic_jax,
    },
    "Sphere": {
        "bounds": (-5.12, 5.12),
        "NumPy": sphere_np,
        "JAX": sphere_jax,
    },
}

ALGORITHMS = {
    "PSO": pso,
    "GDPSO": gdpso,
    "PSO-JAX": pso_jax,
    "GDPSO-JAX": gdpso_jax,
}

DIMS = [2, 10, 30, 50]

HYPERPARAMETERS = {
    "num_dims": None,
    "num_particles": 30,
    "max_iters": 1000,
    "c1": 1.5,
    "c2": 2.5,
    "w": 0.7,
    "seed": None,
    "eta": 0.001,
    "steps": 10,
    "gd_interval": 10,
}

NUM_RUNS = 5

# Experiments

In [ ]:
def run_experiment() -> list[dict]:
    results = []

    total_configs = len(DIMS) * len(ALGORITHMS) * len(BENCHMARKS)
    current_config = 0

    for dim in DIMS:
        print(f"Dimension: {dim}")
        for algorithm_name, algorithm_fn in ALGORITHMS.items():
            backend = "JAX" if "JAX" in algorithm_name else "NumPy"

            for benchmark_name, benchmark_config in BENCHMARKS.items():
                current_config += 1

                objective_fn = benchmark_config[backend]
                bounds = benchmark_config["bounds"]
                hyperparameters = HYPERPARAMETERS.copy()
                hyperparameters["num_dims"] = dim

                print(
                    f"[{current_config}/{total_configs}] Running {algorithm_name} "
                    f"on {benchmark_name}",
                )

                execution_times = []
                fitness_history = []
                for i in range(NUM_RUNS):
                    hyperparameters["seed"] = i

                    if "JAX" in algorithm_name:
                        hyperparameters["seed"] = random.PRNGKey(i)
                        algorithm_fn(objective_fn, bounds, **hyperparameters)

                    start = time.perf_counter()
                    result = algorithm_fn(objective_fn, bounds, **hyperparameters)

                    if "JAX" in algorithm_name:
                        block_until_ready(result)

                    end = time.perf_counter()
                    execution_times.append(end - start)

                    _, fitness, history = result

                    print(f"Iteration {i + 1} | Fitness: {fitness}")
                    fitness_history.append(history)

                mean_fitness_history = jnp.mean(jnp.array(fitness_history), axis=0)
                std_fitness_history = jnp.std(jnp.array(fitness_history), axis=0)

                mean_time = float(jnp.mean(jnp.array(execution_times)))
                std_time = float(jnp.std(jnp.array(execution_times)))

                mean_fitness = float(jnp.mean(mean_fitness_history))
                std_fitness = float(jnp.std(std_fitness_history))

                results.extend(
                    [
                        {
                            "Dimension": dim,
                            "Benchmark": benchmark_name,
                            "Algorithm": algorithm_name,
                            "Execution Time History": execution_times,
                            "Mean of Execution Times (s)": mean_time,
                            "Standard Deviation of Execution Times (s)": std_time,
                            "Mean Fitness History": mean_fitness_history.tolist(),
                            "Std Fitness History": std_fitness_history.tolist(),
                            "Mean of Fitness": mean_fitness,
                            "Standard Deviation of Fitness": std_fitness,
                        },
                    ],
                )

    return results

In [ ]:
results = run_experiment()
df = pd.DataFrame(results)

In [ ]:
df.to_csv(config["output_path"] / "experiment_results.csv")

In [ ]:
generate_visualizations(df, config)